# Data preparation

We will reorganize the files so is easy to work with for the training of our model

## Import libraries

In [ ]:
import random
import shutil
import zipfile

import numpy as np
import tifffile

from zenodo_get import download
from pathlib import Path
from cellpose import io, utils

## Pull dataset from Zenodo

In [ ]:
# Folder that holds A172.zip; the extracted A172/ folder will be created here
data_dir = Path("../data_cp")
data_dir.mkdir(parents=True, exist_ok=True)

if not any(data_dir.rglob("A*.zip")):
        download(
                record_or_doi="10277106",
                output_dir=str(data_dir),
                file_glob="A*.zip",
        )
else:
        print("An A*.zip file already exists; skipping download.")

## Extract and reorganize the dataset

In [ ]:
# Extract only if the dataset is missing or contains no TIFF files.
cell_type = "A172"
root = data_dir / cell_type
zip_path = data_dir / f"{cell_type}.zip"

if not root.exists() or next(root.rglob("*.tif"), None) is None:
    if not zip_path.is_file():
        raise FileNotFoundError(f"Dataset archive not found: {zip_path}")
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(data_dir)
    if next(root.rglob("*.tif"), None) is None:
        raise FileNotFoundError(f"No .tif files were extracted under {root}")
    print(f"Extracted {zip_path} to {root}")
else:
    print(f"{root} already contains TIFF files; skipping extraction.")

### Source dataset layout

The archive contains `train/` and `test/` source splits. Each has an `input/` folder of TIFF images and an `output/` folder of same-named ground-truth masks. We combine both source splits into one pool, make a seeded 80/10/10 sample split, then copy each image-mask pair into its own split folder. The original extracted files are left unchanged.

### Collect matching images and annotations

In [ ]:
# Match each image to its same-named mask across both source splits.
source_pairs = []
seen_image_names = set()

for split_name in ("train", "test"):
    input_dir = root / split_name / "input"
    output_dir = root / split_name / "output"
    if not input_dir.is_dir() or not output_dir.is_dir():
        raise FileNotFoundError(f"Expected input/ and output/ folders under {root / split_name}")

    image_files = sorted(input_dir.glob("*.tif"))
    if not image_files:
        raise FileNotFoundError(f"No .tif images found in {input_dir}")

    for image_path in image_files:
        mask_path = output_dir / image_path.name
        if not mask_path.is_file():
            raise FileNotFoundError(f"No matching annotation for {image_path}")
        if image_path.name in seen_image_names:
            raise ValueError(f"Duplicate image filename across source splits: {image_path.name}")
        seen_image_names.add(image_path.name)
        source_pairs.append((image_path, mask_path))

print(f"Collected {len(source_pairs)} matched image-mask pairs from the source splits")

## Make a reproducible 80/10/10 split

Set `N_IMAGES` to the number of image-mask pairs to sample. It must be a positive multiple of 10 so the train, test, and validation sets remain exactly 80%, 10%, and 10%. `SPLIT_SEED` controls which pairs are selected and keeps the split reproducible. Each image is copied as TIFF, and its annotation is read from TIFF and saved as a NumPy `.npy` array named `<image>_masks.npy` in the same split folder. The original source files are unchanged. Training images are also copied without masks to `A172_manual_training` next to the generated split folder.

In [ ]:
N_IMAGES = 100
SPLIT_SEED = 42

if N_IMAGES <= 0 or N_IMAGES % 10 != 0:
    raise ValueError("N_IMAGES must be a positive multiple of 10 for an exact 80/10/10 split")
if N_IMAGES > len(source_pairs):
    raise ValueError(f"Requested {N_IMAGES} pairs, but only {len(source_pairs)} are available")

selected_pairs = random.Random(SPLIT_SEED).sample(source_pairs, N_IMAGES)
n_train = N_IMAGES * 8 // 10
n_test = N_IMAGES // 10

train_pairs = selected_pairs[:n_train]
test_pairs = selected_pairs[n_train:n_train + n_test]
validation_pairs = selected_pairs[n_train + n_test:]

assert len(train_pairs) == n_train
assert len(test_pairs) == N_IMAGES // 10
assert len(validation_pairs) == N_IMAGES // 10

# Use a separate output root so the original A172/train and A172/test remain untouched.
prepared_dir = data_dir / f"{cell_type}_splits_n{N_IMAGES}_seed{SPLIT_SEED}"
split_groups = {
    "train": train_pairs,
    "test": test_pairs,
    "validation": validation_pairs,
}

for split_name, pairs in split_groups.items():
    split_dir = prepared_dir / split_name
    split_dir.mkdir(parents=True, exist_ok=True)
    copied_pairs = []
    for image_path, mask_path in pairs:
        split_image = split_dir / image_path.name
        split_mask = split_dir / f"{image_path.stem}_seg.npy"
        shutil.copy2(image_path, split_image)
        mask_array = io.imread(str(mask_path)).squeeze().astype(np.uint16)
        # otlines must be also added to the dictionary otherwise cellpose does not read the mask.
        outlines = mask_array * utils.masks_to_outlines(mask_array)
        if mask_array.ndim != 2 or not np.issubdtype(mask_array.dtype, np.integer):
            raise ValueError(f"Expected a 2D integer-label annotation: {mask_path}")
        # print(f'"split_mask": {split_mask}, "filename": {str(image_path)}')
        np.save(split_mask, {"masks": mask_array, "outlines": outlines, "filename": str(image_path)})
        copied_pairs.append((split_image, split_mask))
    split_groups[split_name] = copied_pairs

manual_training_dir = data_dir / f"{cell_type}_manual_training"
manual_training_dir.mkdir(parents=True, exist_ok=True)
for split_image, _ in split_groups["train"]:
    shutil.copy2(split_image, manual_training_dir / split_image.name)

train_pairs = split_groups["train"]
test_pairs = split_groups["test"]
validation_pairs = split_groups["validation"]

assert len(train_pairs) == n_train
assert len(test_pairs) == N_IMAGES // 10
assert len(validation_pairs) == N_IMAGES // 10
print(f"Copied {len(train_pairs)} train, {len(test_pairs)} test, and {len(validation_pairs)} validation pairs to {prepared_dir}")
print(f"Copied {len(split_groups['train'])} training images without masks to {manual_training_dir}")

### Launch the cellpose GUI and explore the images

In [ ]:
# import sys
# from cellpose.gui import gui
# app = gui.run(); sys.exit(app.exec_())